<a href="https://colab.research.google.com/github/revadirevathi20-sys/hdb-resale-etl-pipeline/blob/main/notebooks/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HDB Resale Flat Price

This notebook explores the combined HDB resale dataset before the ETL pipeline is run.
The goal is to understand the data's structure, distributions, and potential quality issu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

%matplotlib inline

## 1. Load Data

In [2]:
df = pd.read_csv("../data/raw/master_combined_raw.csv")
print(f"Shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/master_combined_raw.csv'

## 2. Basic Data Profiling

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Null value counts
null_counts = df.isnull().sum().sort_values(ascending=False)
print("Null counts:\n", null_counts)

In [ ]:
# Duplicate rows
print(f"Total duplicate rows: {df.duplicated().sum()}")

## 3. Date Distribution

Check the spread of transactions over time.

In [ ]:
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")
df["year"] = df["month"].dt.year

transactions_by_year = df.groupby("year").size()

plt.figure(figsize=(12, 4))
transactions_by_year.plot(kind="bar", color="steelblue")
plt.title("Number of Transactions by Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Town Distribution

In [ ]:
town_counts = df["town"].value_counts()

plt.figure(figsize=(14, 5))
town_counts.plot(kind="bar", color="teal")
plt.title("Number of Transactions by Town")
plt.xlabel("Town")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Flat Type Distribution

In [ ]:
flat_type_counts = df["flat_type"].value_counts()

plt.figure(figsize=(8, 4))
flat_type_counts.plot(kind="bar", color="coral")
plt.title("Number of Transactions by Flat Type")
plt.xlabel("Flat Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Resale Price Distribution

Understanding the spread and identifying potential outliers.

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["resale_price"], bins=100, kde=True, color="steelblue")
plt.title("Distribution of Resale Prices")
plt.xlabel("Resale Price (SGD)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot by flat type
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="flat_type", y="resale_price", palette="Set2")
plt.title("Resale Price by Flat Type")
plt.xlabel("Flat Type")
plt.ylabel("Resale Price (SGD)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Potential price outliers using IQR
Q1 = df["resale_price"].quantile(0.25)
Q3 = df["resale_price"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["resale_price"] < lower) | (df["resale_price"] > upper)]
print(f"Potential price outliers (IQR method): {len(outliers)}")
print(f"Lower bound: {lower:,.0f} | Upper bound: {upper:,.0f}")
outliers[["month", "town", "flat_type", "resale_price"]].sort_values("resale_price", ascending=False).head(20)

## 7. Floor Area Distribution

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df["floor_area_sqm"], bins=60, kde=True, color="mediumseagreen")
plt.title("Distribution of Floor Area (sqm)")
plt.xlabel("Floor Area (sqm)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Flag suspicious floor areas
suspicious_area = df[(df["floor_area_sqm"] < 20) | (df["floor_area_sqm"] > 400)]
print(f"Suspicious floor area records: {len(suspicious_area)}")
suspicious_area[["month", "town", "flat_type", "floor_area_sqm"]].head(10)

## 8. Storey Range Check

In [ ]:
# Check format consistency of storey_range
invalid_storey = df[~df["storey_range"].str.match(r"^\d{2} TO \d{2}$", na=False)]
print(f"Invalid storey_range format: {len(invalid_storey)}")
invalid_storey[["storey_range"]].value_counts().head(10)

## 9. Lease Commence Date

In [ ]:
lease_counts = df["lease_commence_date"].value_counts().sort_index()

plt.figure(figsize=(14, 4))
lease_counts.plot(kind="bar", color="slateblue")
plt.title("Transactions by Lease Commence Year")
plt.xlabel("Lease Commence Year")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Average Resale Price Trend Over Time

In [ ]:
avg_price_by_month = df.groupby("month")["resale_price"].mean()

plt.figure(figsize=(14, 5))
avg_price_by_month.plot(color="tomato")
plt.title("Average Resale Price Over Time")
plt.xlabel("Month")
plt.ylabel("Average Resale Price (SGD)")
plt.tight_layout()
plt.show()

## 11. EDA Summary

- Date range spans from **2012-01 to December 2016. **
- Most transactions are in towns like **Tampines, Woodlands, Jurong West**
- **4-room flats** are the most common flat type
- Resale prices show a clear upward trend post-2020
- A small number of potential price outliers were identified using the IQR method
- No major issues found with storey_range format
- Floor area values appear within expected ranges